# Fine-Tuning LLMs

In this exercise, you will fine-tune the [Flan-T5](https://huggingface.co/docs/transformers/model_doc/flan-t5) model for enhanced dialogue summarization. You will first explore a full fine-tuning approach and evaluate the results with ROUGE metrics. Then you will perform Parameter-Efficient Fine-Tuning (PEFT), evaluate the resulting model and see that the benefits of PEFT outweigh the slightly-lower performance metrics.

## 1. Set up Dependencies and Load Dataset and LLM

In [1]:
!pip install datasets evaluate rouge_score peft -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00


In [2]:
import torch
import time
import pandas as pd
import numpy as np

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, Seq2SeqTrainer
from datasets import load_dataset

In [3]:
dataset = load_dataset('knkarthick/dialogsum')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Load the pre-trained [Flan-T5 model](https://huggingface.co/docs/transformers/model_doc/flan-t5) and its tokenizer from HuggingFace. Notice that you will be using the [small version](https://huggingface.co/google/flan-t5-base) of Flan-T5. Setting `torch_dtype=torch.bfloat16` specifies the data type to be used by this model, which can reduce GPU memory usage since `bfloat16` uses half as much memory per number compared to `float32`, the default precision for most models.

In [4]:
model_name = 'google/flan-t5-base'

original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

## 2. Test the Model with Zero-Shot Inferencing

Test the model with zero-shot inference.

In [5]:
index = 42
dash_line = '-' * 100

dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

prompt = f"Summarize the following conversation.\n{dialogue}\nSummary:\n"
inputs = tokenizer(prompt, return_tensors='pt')
output = original_model.generate(inputs['input_ids'], max_new_tokens=50)[0]
original_model_summary = tokenizer.decode(output, skip_special_tokens=True)

print(dash_line)
print(f'INPUT PROMPT:\n{dialogue}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{original_model_summary}\n')

----------------------------------------------------------------------------------------------------
INPUT PROMPT:
#Person1#: I don't know how to adjust my life. Would you give me a piece of advice?
#Person2#: You look a bit pale, don't you?
#Person1#: Yes, I can't sleep well every night.
#Person2#: You should get plenty of sleep.
#Person1#: I drink a lot of wine.
#Person2#: If I were you, I wouldn't drink too much.
#Person1#: I often feel so tired.
#Person2#: You better do some exercise every morning.
#Person1#: I sometimes find the shadow of death in front of me.
#Person2#: Why do you worry about your future? You're very young, and you'll make great contribution to the world. I hope you take my advice.
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# wants to adjust #Person1#'s life and #Person2# suggests #Person1# be positive and stay healthy.
-------------------------------------------------------

You can see that the model struggles to summarize the dialogue compared to the baseline summary, and simply repeats the first sentence from the dialogue.

## 3. Perform Full Fine-Tuning

### 3.1 Preprocess the Dataset

You need to convert the dialog-summary (prompt-response) pairs into explicit instructions for the LLM. Prepend an instruction to the start of the dialog with `Summarize the following conversation.`, and to the start of the summary with `Summary:` as follows:

Training prompt (dialogue):
```
Summarize the following conversation.
Alice: This is her part of the conversation.
Bob: This is his part of the conversation.    
Summary:
```

Training response (summary):
```
Both Alice and Bob participated in the conversation.
```

**Exercise**: Write a function to tokenize a batch of examples from the dialogue dataset. The function should concatentate the dialogues with the predefined prompt, tokenize them along with their summaries, and define the tokenized summaries as the labels.

In [6]:
def tokenize(examples):
    ### WRITE YOUR CODE HERE
    # Add prompt to each dialogue
    inputs = ["Summarize the following conversation.\n" + dialogue + "\nSummary:"
              for dialogue in examples["dialogue"]]

    # Tokenize inputs and targets
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    # Tokenize summaries
    labels = tokenizer(examples["summary"], max_length=128, truncation=True)

    # Set the labels as the model's targets
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [7]:
tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

### 3.2 Fine-Tune the Model

**Exercise**: Utilize the Hugging Face Trainer API for training the model on the preprocessed dataset. Define the training arguments, a data collator, and create a `Seq2SeqTrainer` instance. Train the model for one epoch.

In [8]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `Akshara-mistral` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Ak

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Huggingface token")

In [ ]:
from huggingface_hub import login
login(secret_value_0)

In [ ]:
### WRITE YOUR CODE HERE
# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="AksharaReddy18/Finetuning_samsung",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,  # Keep only the latest 2 checkpoints
    learning_rate=5e-5,
    push_to_hub=True,
    hub_model_id="AksharaReddy18/Finetuning_samsung",
    hub_strategy="every_save",
    report_to="tensorboard",
)

# Create a data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=original_model,
    padding=True
)

# Create the trainer
trainer = Seq2SeqTrainer(
    model=original_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

<ipython-input-10-33df2265b138>:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Training a fully fine-tuned version of the model should take about 10 minutes on a Google Colab GPU machine.

In [ ]:
# Train and push
try:
    trainer.train()
finally:
    trainer.push_to_hub()  # Push even if training crashes

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
100,2.016700
200,1.852000
300,1.658600
400,1.496200
500,1.449000
600,1.370000
700,1.444300
800,1.422100
900,1.397300
1000,1.363900


model.safetensors:   0%|          | 0.00/495M [00:00<?, ?B/s]

Save the model to a local folder:

In [9]:
model_path = './flan-t5-base-dialogsum-checkpoint'

original_model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

('./flan-t5-base-dialogsum-checkpoint/tokenizer_config.json',
 './flan-t5-base-dialogsum-checkpoint/special_tokens_map.json',
 './flan-t5-base-dialogsum-checkpoint/spiece.model',
 './flan-t5-base-dialogsum-checkpoint/added_tokens.json',
 './flan-t5-base-dialogsum-checkpoint/tokenizer.json')

Create an instance of the `AutoModelForSeq2SeqLM` class for the instruct model:

In [10]:
# I have already trained the model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Replace with your repo ID
repo_id = "AksharaReddy18/Finetuning_samsung"

# Load the trained model and tokenizer
instruct_model = AutoModelForSeq2SeqLM.from_pretrained(repo_id)
tokenizer = AutoTokenizer.from_pretrained(repo_id)

config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/495M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Reload the original Flan-T5-base model:

In [11]:
original_model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base', torch_dtype=torch.bfloat16)

### 3.3 Evaluate the Model Qualitatively (Human Evaluation)

**Exercise**: Make inferences for the same example as in Section 2, using the original model and the fully fine-tuned model.

In [12]:
### WRITE YOUR CODE HERE
index = 42
dash_line = '-' * 100

dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

prompt = f"Summarize the following conversation.\n{dialogue}\nSummary:\n"
inputs = tokenizer(prompt, return_tensors='pt')

# Generate summaries with both models
original_output = original_model.generate(inputs['input_ids'], max_new_tokens=50)[0]
instruct_output = instruct_model.generate(inputs['input_ids'], max_new_tokens=50)[0]

# Decode the summaries
original_model_summary = tokenizer.decode(original_output, skip_special_tokens=True)
instruct_model_summary = tokenizer.decode(instruct_output, skip_special_tokens=True)

# Print the results
print(dash_line)
print(f'INPUT PROMPT:\n{dialogue}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{original_model_summary}')
print(dash_line)
print(f'MODEL GENERATION - FINE-TUNED:\n{instruct_model_summary}')


----------------------------------------------------------------------------------------------------
INPUT PROMPT:
#Person1#: I don't know how to adjust my life. Would you give me a piece of advice?
#Person2#: You look a bit pale, don't you?
#Person1#: Yes, I can't sleep well every night.
#Person2#: You should get plenty of sleep.
#Person1#: I drink a lot of wine.
#Person2#: If I were you, I wouldn't drink too much.
#Person1#: I often feel so tired.
#Person2#: You better do some exercise every morning.
#Person1#: I sometimes find the shadow of death in front of me.
#Person2#: Why do you worry about your future? You're very young, and you'll make great contribution to the world. I hope you take my advice.
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# wants to adjust #Person1#'s life and #Person2# suggests #Person1# be positive and stay healthy.
-------------------------------------------------------

The fine-tuned model is able to create a much better summary of the dialogue compared to the original model.

### 3.4 Evaluate the Model Quantitatively (with ROUGE Metric)

The [ROUGE metric](https://en.wikipedia.org/wiki/ROUGE_(metric)) helps quantify the validity of summarizations produced by models. It compares summarizations to a "baseline" summary which is usually created by a human. While not perfect, it does indicate the overall increase in summarization effectiveness that we have accomplished by fine-tuning.

**Exercise**: Generate the outputs for a sample of the test set with the fine-tuned model (use only the first 10 dialogues and summaries to save time).

In [13]:
### WRITE YOUR CODE HERE
# Generate summaries for a sample of the test set
sample_size = 10

# Create lists to store the outputs
human_baseline_summaries = []
original_model_summaries = []
instruct_model_summaries = []

# Loop through the first 'sample_size' examples
for i in range(sample_size):
    # Get the dialogue and summary
    dialogue = dataset['test'][i]['dialogue']
    summary = dataset['test'][i]['summary']

    # Add the human baseline summary
    human_baseline_summaries.append(summary)

    # Create the prompt
    prompt = f"Summarize the following conversation.\n{dialogue}\nSummary:\n"
    inputs = tokenizer(prompt, return_tensors='pt')

    # Generate summaries with both models
    original_output = original_model.generate(inputs['input_ids'], max_new_tokens=50)[0]
    instruct_output = instruct_model.generate(inputs['input_ids'], max_new_tokens=50)[0]

    # Decode the summaries
    original_summary = tokenizer.decode(original_output, skip_special_tokens=True)
    instruct_summary = tokenizer.decode(instruct_output, skip_special_tokens=True)

    # Add the summaries to the lists
    original_model_summaries.append(original_summary)
    instruct_model_summaries.append(instruct_summary)

    # Print progress
    print(f"Processed example {i+1}/{sample_size}")

# Print a sample comparison
print("\nSample comparison (example 0):")
print(f"Human: {human_baseline_summaries[0]}")
print(f"Original: {original_model_summaries[0]}")
print(f"Fine-tuned: {instruct_model_summaries[0]}")


Processed example 1/10
Processed example 2/10
Processed example 3/10
Processed example 4/10
Processed example 5/10
Processed example 6/10
Processed example 7/10
Processed example 8/10
Processed example 9/10
Processed example 10/10

Sample comparison (example 0):
Human: Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.
Original: #Person1#: I need to take a dictation for you.
Fine-tuned: #Person1# asks Ms. Dawson to take a dictation for #Person2#. Ms. Dawson says that all office communications are restricted to email correspondence and official memos. Ms. Dawson says that


Evaluate the models computing ROUGE metrics:

In [14]:
from rouge_score import rouge_scorer

def compute_rouge_scores(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores_list = [scorer.score(ref, pred) for pred, ref in zip(predictions, references)]

    avg_scores = {}
    for key in ['rouge1', 'rouge2', 'rougeL']:
        avg_scores[key] = sum(score[key].fmeasure for score in scores_list) / len(scores_list)

    return avg_scores

# Compute results
original_model_results = compute_rouge_scores(
    original_model_summaries,
    human_baseline_summaries[:len(original_model_summaries)]
)

instruct_model_results = compute_rouge_scores(
    instruct_model_summaries,
    human_baseline_summaries[:len(instruct_model_summaries)]
)

# Print
print('ORIGINAL MODEL:')
print(original_model_results)

print('INSTRUCT MODEL:')
print(instruct_model_results)

ORIGINAL MODEL:
{'rouge1': 0.2578478599350692, 'rouge2': 0.12061206412575634, 'rougeL': 0.2368848969721063}
INSTRUCT MODEL:
{'rouge1': 0.46207076125303936, 'rouge2': 0.1605716252628898, 'rougeL': 0.3505359237870258}


The results show substantial improvement in all ROUGE metrics:

In [15]:
print("Absolute percentage improvement of the instruct model over the original model:")

for key in instruct_model_results:
    improvement = instruct_model_results[key] - original_model_results[key]
    print(f'{key}: {improvement*100:.2f}%')

Absolute percentage improvement of the instruct model over the original model:
rouge1: 20.42%
rouge2: 4.00%
rougeL: 11.37%


The results indicate that instruction tuning significantly enhanced the model’s ability to generate more relevant, accurate, and well-structured summaries.

## 4. Perform Parameter Efficient Fine-Tuning (PEFT)

Now, let's perform **Parameter Efficient Fine-Tuning (PEFT)** instead of "full fine-tuning" as you did above. PEFT is a form of instruction fine-tuning that is much more efficient than full fine-tuning, with comparable evaluation results as you will see soon.

One of the most popular PEFT methods is **Low-Rank Adaptation (LoRA)**, which  introduces low-rank matrices to adapt the LLM with minimal additional parameters. In most cases, when someone says PEFT, they typically mean LoRA.  After fine-tuning for a specific task with LoRA, the result is that the original LLM remains unchanged and a newly-trained "LoRA adapter" emerges. This LoRA adapter is much smaller than the original LLM - on the order of a single-digit % of the original LLM size (MBs vs GBs).  

At inference time, the LoRA adapter is reunited and combined with its original LLM to serve the inference request. The benefit is that many LoRA adapters can re-use the original LLM which reduces overall memory requirements when serving multiple tasks and use cases.

### 4.1 Setup the LoRA model for Fine-Tuning

You first need to define the configuration of the LoRA model. Have a look at the configuration below. The key configuration element to adjust is the rank (`r`) of the adapter, which influences its capacity and complexity. Experiment with various ranks, such as 8, 16, or 32, and see how they affect the results.

In [16]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,
    lora_alpha=32,
    lora_dropout=0.1
)

Add LoRA adapter layers/parameters to the original LLM to be trained:

In [17]:
peft_model = get_peft_model(original_model, lora_config)

The number of trainable model parameters in the LoRA model is:

In [18]:
peft_model.print_trainable_parameters()

trainable params: 3,538,944 || all params: 251,116,800 || trainable%: 1.4093


### 4.2 Train the LoRA Adapter

**Exercise**: Define training arguments and create a `Seq2SeqTrainer` instance for the LoRA model. Use a higher learning rate than full fine-tuning (e.g., `1e-3`).

In [ ]:
### WRITE YOUR CODE HERE

# Define training arguments for PEFT
peft_training_args = Seq2SeqTrainingArguments(
    output_dir="AksharaReddy18/PEFT-training",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./peft_logs",
    fp16=False,
    logging_steps=100,
    push_to_hub=True,       # This is the key parameter to push to Hub
    hub_model_id="AksharaReddy18/PEFT-training",  # Same as output_dir
    hub_strategy="every_save",
    save_strategy="epoch",
    learning_rate=1e-3  # Higher learning rate for PEFT
)

# Create a data collator
peft_data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=peft_model,
    padding=True
)

# Create the trainer
peft_trainer = Seq2SeqTrainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_dataset["train"],
    tokenizer=tokenizer,
    data_collator=peft_data_collator
)

/tmp/ipykernel_31/186735187.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  peft_trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Train the PEFT adapter. Training should take about 6 minutes on a Google Colab GPU machine.

In [ ]:
import wandb
# Retrieve Wandb API key from Kaggle Secrets
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("Wandb key")

# Log into Wandb
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akshara-patlu1803 (akshara-patlu1803-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
peft_trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,1.712400
200,1.360000
300,1.294000
400,1.346200
500,1.312100
600,1.306000
700,1.281800
800,1.257600
900,1.221800
1000,1.235500


TrainOutput(global_step=3115, training_loss=1.2275030096308186, metrics={'train_runtime': 1277.5176, 'train_samples_per_second': 9.753, 'train_steps_per_second': 2.438, 'total_flos': 5924601444519936.0, 'train_loss': 1.2275030096308186, 'epoch': 1.0})

Save the model to a local folder:

In [19]:
# I have already trained the model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Replace with your repo ID
repo_id = "AksharaReddy18/PEFT-training"

# Load the trained model and tokenizer
peft = AutoModelForSeq2SeqLM.from_pretrained(repo_id)
tokenizer = AutoTokenizer.from_pretrained(repo_id)

adapter_config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/14.2M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

In [28]:
peft.save_pretrained('./flan-t5-base-dialogsum-lora')

Load the PEFT model:

In [29]:
from peft import AutoPeftModelForSeq2SeqLM
from transformers import AutoTokenizer

peft_model = AutoModelForSeq2SeqLM.from_pretrained('./flan-t5-base-dialogsum-lora')
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')

Reload the original Flan-T5-base model:

In [30]:
original_model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base', torch_dtype=torch.bfloat16)

### 4.3 Evaluate the Model Qualitatively (Human Evaluation)

**Exercise**: Make inferences for the same example as in Sections 2 and 3, using the original model, the fully fine-tuned model and the PEFT model.

In [31]:
### WRITE YOUR CODE HERE
index = 42
dash_line = '-' * 100

dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

prompt = f"Summarize the following conversation.\n{dialogue}\nSummary:\n"
inputs = tokenizer(prompt, return_tensors='pt')

# Generate summaries with both models
original_output = original_model.generate(inputs['input_ids'], max_new_tokens=50)[0]
instruct_output = instruct_model.generate(inputs['input_ids'], max_new_tokens=50)[0]
peft_output = peft_model.generate(inputs['input_ids'], max_new_tokens=50)[0]

# Decode the summaries
original_model_summary = tokenizer.decode(original_output, skip_special_tokens=True)
instruct_model_summary = tokenizer.decode(instruct_output, skip_special_tokens=True)
peft_model_summary = tokenizer.decode(peft_output, skip_special_tokens=True)

# Print the results
print(dash_line)
print(f'INPUT PROMPT:\n{dialogue}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{original_model_summary}')
print(dash_line)
print(f'MODEL GENERATION - FINE-TUNED:\n{instruct_model_summary}')
print(dash_line)
print(f'MODEL GENERATION - PEFT:\n{peft_model_summary}')

----------------------------------------------------------------------------------------------------
INPUT PROMPT:
#Person1#: I don't know how to adjust my life. Would you give me a piece of advice?
#Person2#: You look a bit pale, don't you?
#Person1#: Yes, I can't sleep well every night.
#Person2#: You should get plenty of sleep.
#Person1#: I drink a lot of wine.
#Person2#: If I were you, I wouldn't drink too much.
#Person1#: I often feel so tired.
#Person2#: You better do some exercise every morning.
#Person1#: I sometimes find the shadow of death in front of me.
#Person2#: Why do you worry about your future? You're very young, and you'll make great contribution to the world. I hope you take my advice.
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# wants to adjust #Person1#'s life and #Person2# suggests #Person1# be positive and stay healthy.
-------------------------------------------------------

The zero-shot model generation focuses only on Person1’s initial request without summarizing Person2's advice. The fine-tuned model generation offers a more detailed summary, noting Person1's concern about the future and Person2’s advice on sleep and exercise. The PEFT model generation gives a simpler summary, emphasizing Person1’s request and Person2’s health-related advice but without mentioning the emotional concerns about the future.

### 4.4 Evaluate the Model Quantitatively (with ROUGE Metric)

**Exercise**: Generate the outputs for a sample of the test set with the PEFT model (use only the first 10 dialogues and summaries to save time).

In [32]:
### WRITE YOUR CODE HERE

# Generate summaries for a sample of the test set
sample_size = 10

# Create lists to store the outputs
human_baseline_summaries = []
original_model_summaries = []
instruct_model_summaries = []
peft_model_summaries= []

# Loop through the first 'sample_size' examples
for i in range(sample_size):
    # Get the dialogue and summary
    dialogue = dataset['test'][i]['dialogue']
    summary = dataset['test'][i]['summary']

    # Add the human baseline summary
    human_baseline_summaries.append(summary)

    # Create the prompt
    prompt = f"Summarize the following conversation.\n{dialogue}\nSummary:\n"
    inputs = tokenizer(prompt, return_tensors='pt')

    # Generate summaries with all three models
    original_output = original_model.generate(inputs['input_ids'], max_new_tokens=50)[0]
    instruct_output = instruct_model.generate(inputs['input_ids'], max_new_tokens=50)[0]
    peft_output = peft_model.generate(inputs['input_ids'], max_new_tokens=50)[0]

    # Decode the summaries
    original_summary = tokenizer.decode(original_output, skip_special_tokens=True)
    instruct_summary = tokenizer.decode(instruct_output, skip_special_tokens=True)
    peft_summary = tokenizer.decode(peft_output, skip_special_tokens=True)

    # Add the summaries to the lists
    original_model_summaries.append(original_summary)
    instruct_model_summaries.append(instruct_summary)
    peft_model_summaries.append(peft_summary)

    # Print progress
    print(f"Processed example {i+1}/{sample_size}")

# Print a sample comparison
print("\nSample comparison (example 0):")
print(f"Human: {human_baseline_summaries[0]}")
print(f"Original: {original_model_summaries[0]}")
print(f"Fine-tuned: {instruct_model_summaries[0]}")
print(f"PEFT: {peft_model_summaries[0]}")

Processed example 1/10
Processed example 2/10
Processed example 3/10
Processed example 4/10
Processed example 5/10
Processed example 6/10
Processed example 7/10
Processed example 8/10
Processed example 9/10
Processed example 10/10

Sample comparison (example 0):
Human: Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.
Original: #Person1#: I need to take a dictation for you.
Fine-tuned: #Person1# asks Ms. Dawson to take a dictation for #Person2#. Ms. Dawson says that all office communications are restricted to email correspondence and official memos. Ms. Dawson says that
PEFT: #Person1# asks Ms. Dawson to take a dictation for #Person1#. Ms. Dawson tells #Person1# the new policy applies to internal and external communications.


Compute ROUGE score for this subset of the data.

In [33]:
from rouge_score import rouge_scorer

def compute_rouge_scores(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores_list = [scorer.score(ref, pred) for pred, ref in zip(predictions, references)]

    avg_scores = {}
    for key in ['rouge1', 'rouge2', 'rougeL']:
        avg_scores[key] = sum(score[key].fmeasure for score in scores_list) / len(scores_list)

    return avg_scores

# Compute results
original_model_results = compute_rouge_scores(
    original_model_summaries,
    human_baseline_summaries[:len(original_model_summaries)]
)

instruct_model_results = compute_rouge_scores(
    instruct_model_summaries,
    human_baseline_summaries[:len(instruct_model_summaries)]
)

peft_model_results = compute_rouge_scores(
    peft_model_summaries,
    human_baseline_summaries[:len(peft_model_summaries)]
)
# Print
print('ORIGINAL MODEL:')
print(original_model_results)

print('INSTRUCT MODEL:')
print(instruct_model_results)

print('PEFT MODEL:')
print(peft_model_results)

ORIGINAL MODEL:
{'rouge1': 0.2578478599350692, 'rouge2': 0.12061206412575634, 'rougeL': 0.2368848969721063}
INSTRUCT MODEL:
{'rouge1': 0.46207076125303936, 'rouge2': 0.1605716252628898, 'rougeL': 0.3505359237870258}
PEFT MODEL:
{'rouge1': 0.43938663933797456, 'rouge2': 0.16524825577457158, 'rougeL': 0.3563005303829985}


Notice, that PEFT model results are not too bad, while the training process was much easier!

Calculate the improvement of PEFT over the original model:

In [34]:
print("Absolute percentage improvement of the PEFT model over the original model:")

for key in peft_model_results:
    improvement = peft_model_results[key] - original_model_results[key]
    print(f'{key}: {improvement*100:.2f}%')

Absolute percentage improvement of the PEFT model over the original model:
rouge1: 18.15%
rouge2: 4.46%
rougeL: 11.94%


Overall, the PEFT model generates more accurate and coherent summaries, with noticeable improvements in content overlap and structure.

Now calculate the improvement of PEFT over a full fine-tuned model:

In [35]:
print("Absolute percentage improvement of the PEFT model over the instruct model:")

for key in peft_model_results:
    improvement = peft_model_results[key] - instruct_model_results[key]
    print(f'{key}: {improvement*100:.2f}%')

Absolute percentage improvement of the PEFT model over the instruct model:
rouge1: -2.27%
rouge2: 0.47%
rougeL: 0.58%


You can see a small percentage decrease in the ROUGE metrics vs. full fine-tuned. However, the training requires much less computing and memory resources.

- The PEFT model shows significant improvement over the original model, with ROUGE-1, ROUGE-2, and ROUGE-L scores increasing by 18.15%, 4.46%, and 11.94%, respectively, indicating better content overlap and sequence preservation. However, when compared to the Instruct model, the PEFT model shows a slight decline in ROUGE-1 (-2.27%) but modest improvements in ROUGE-2 (0.47%) and ROUGE-L (0.58%), suggesting similar performance with minor gains in structural aspects.
- Sample comparisons highlight that the PEFT model generates more detailed and contextually relevant summaries, offering a clearer understanding of the policy change regarding Instant Messaging usage, which the original model fails to capture.
- Overall, the PEFT model significantly enhances summary quality over the original, while performing comparably to the Instruct model.